#Exercícios Complementares

Carregue o dataset a seguir.

In [ ]:
import requests, os


filename = 'b2w.csv'

def download (filename):
  url = f'https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/{filename}'

  if (os.path.isfile(f'./{filename}')):
    print('Arquivo já existente no Runtime... Tudo OK')
    return

  response = requests.get(url)
  with open(f'./{filename}', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')

download(filename)



Download realizado e arquivo extraído no Runtime... Tudo OK


Nesse exercício, vamos utilizar técnicas de aprendizado não supervisionado para analisar um dataset rotulado de análise de sentimento na plataforma B2W.

## Separação dos dados e Pré-processamento
1. Implemente uma função de remoção de stopwords e lematização dos termos;
2. Separe os dados de acordo com seu rótulo de sentimento (positivo e negativo), considerando o atributo polarity, e selecione apenas os documentos rotulados com sentimento negativo (polarity==0).

**Resultado**:
O córpus resultante terá N documentos e os 3 documentos pré-processados do corpus negativo seriam:
* 'recebir produto prazo vir defeito trar ser Americanas resolver preciso produto'
* 'comprar produto , vir botão travar ligar ligar . Péssima qualidade .'
* 'azar , produto ligar . entregar fornecedor .'





In [ ]:
!pip install spacy
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 49.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd, spacy


df = pd.read_csv('./b2w.csv').dropna()
corpus_positive = df.loc[df['polarity'] == 1, 'review_text']
corpus_negative = df.loc[df['polarity'] == 0, 'review_text']

nlp = spacy.load('pt_core_news_sm', disable=["morphologizer", "senter", "attribute_ruler", "ner"])


def preprocess(corpus):
  result = []

  for text in corpus.tolist():
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    result.append(' '.join(tokens))

  return result


corpus_negative_preprocessed = preprocess(corpus_negative)

print(len(corpus_negative_preprocessed))
print(corpus_negative_preprocessed[0:3])

35758
['recebir produto prazo vir defeito trar ser Americanas resolver preciso produto', 'comprar produto , vir botão travar ligar ligar . Péssima qualidade .', 'azar , produto ligar . entregar fornecedor .']


## TF-IDF e K-Means
1. Realize a extração de características utilizando TF-IDF do corpus (documentos de sentimento negativo);
2. Treine um modelo K-Means com 100 clusters e random_state=42, utilizando essas características;
3. Identifique a qual cluster o documento de teste pertence e apresente 5 documentos desse cluster, para analisar a temática dos documentos desse cluster.

**Resultado:**
O id do cluster ao qual o documento teste pertence é 7 e os 5 primeiros documentos desse cluster são:
- Recomendo que me devolvam o dinheiro e me mande uma carta pedindo desculpa pela LIXO que eu comprei. Sacanagem e enganação. Nao sei como a americanas deixa oferecer um produto de LIXO como isso ai...LIXO LIXO LIXO  Isso parece uma meia velha mesmo...alguem ja falou isso ai embaixo..LIXO
- ESSE PRODUTO NÃO CHEGOU EM MINHA RESIDÊNCIA!!!!!! Descontaram o valor em meu cartão, o produto não chegou em minha residência.  Fui noite dos correios e está com endereço final sendo SÃO PAULO, porém meu endereço é no RIO DE JANEIRO.  Quero meu dinheiro de volta ou o produto!!
- Produto péssima qualidade, nao gostei. Quero meu dinheiro de volta.
- Esse produto é muito ruim , gastasse muito café e fora que para preparar é uma briga   Não compre, não percam o seu dinheiro
- comprei o meu e nunca funcionou...tentei trocar, entrar em contato com a assistência, simplesmente não respondem...uma porcaria, nao comprem, é dinheiro jogado fora.

In [ ]:
test_text = 'Jogo incompatível com o console pedido. Dinheiro jogado fora.'

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans


vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus_negative_preprocessed)

kmeans = KMeans(n_clusters=100, random_state=42)
kmeans.fit(X)

X_test = vectorizer.transform([test_text])
cluster_label = kmeans.predict(X_test)[0]

print(f'O documento "{test_text}" pertence ao cluster {cluster_label}.')

print('5 primeioros documentos desse cluster são:')
count = 0
for i, text in enumerate(corpus_negative):
  if kmeans.labels_[i] == cluster_label:
    print(f'- {text}')
    count += 1
    if count == 5:
      break





/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


O documento "Jogo incompatível com o console pedido. Dinheiro jogado fora." pertence ao cluster 16.
5 primeioros documentos desse cluster são:
- Recomendo que me devolvam o dinheiro e me mande uma carta pedindo desculpa pela LIXO que eu comprei. Sacanagem e enganação. Nao sei como a americanas deixa oferecer um produto de LIXO como isso ai...LIXO LIXO LIXO  Isso parece uma meia velha mesmo...alguem ja falou isso ai embaixo..LIXO
- ESSE PRODUTO NÃO CHEGOU EM MINHA RESIDÊNCIA!!!!!! Descontaram o valor em meu cartão, o produto não chegou em minha residência.  Fui noite dos correios e está com endereço final sendo SÃO PAULO, porém meu endereço é no RIO DE JANEIRO.  Quero meu dinheiro de volta ou o produto!!
- Produto péssima qualidade, nao gostei. Quero meu dinheiro de volta.
- Esse produto é muito ruim , gastasse muito café e fora que para preparar é uma briga   Não compre, não percam o seu dinheiro
- comprei o meu e nunca funcionou...tentei trocar, entrar em contato com a assistência, si

##CountVectorizer e LDA
1. Realize a extração de características utilizando CountVectorizer binário do corpus (documentos de sentimento negativo);
2. Treine um modelo LDA com 100 componentes e random_state=42, utilizando essas características;
3. Identifique os 3 tópicos que apresentam maior compatibilidade com o documento de teste pertence e apresente as 10 palavras que possuem maior influência na determinação desses tópicos.

**Resultado:**
- Tópico 0 (com probabilidade de 0.2556332015292843): iphone, foto, 32, cartão, celular, película, memória, gb, livro, capa
- Tópico 5 (com probabilidade de 0.2597627919293714): chegar, comprei, enganoso, compr, foto, diferente, comprar, propaganda, vir, produto
- Tópico 18 (com probabilidade de 0.3901595603367901): solicitar, resposta, enviar, americanas, retorno, defeito, vir, troca, devolução, produto

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA



extractor = CountVectorizer(binary=True)
X = extractor.fit_transform(corpus_negative_preprocessed)
model = LDA(n_components=100, random_state=42)
model.fit(X)



LatentDirichletAllocation(n_components=20, random_state=42)

In [ ]:
X_test = extractor.transform([test_text])
topic_probabilities = model.transform(X_test)[0]
top5_topics = topic_probabilities.argsort()[-5:]
word_index = extractor.get_feature_names_out()

print('Os 5 tópicos que possuem maior compatibilidade com o documento de teste são:')
for topic in top5_topics:
  top10_words = model.components_[topic].argsort()[-10:]
  words = ', '.join([ word_index[i] for i in top10_words ])
  print(f'- Tópico {topic} ({topic_probabilities[topic]}): {words}')



Os 5 tópicos que possuem maior compatibilidade com o documento de teste são:
- Tópico 12 (0.005555555765489102): ja, chegar, nao, compr, comprar, entrega, produto, dia, fiscal, nota
- Tópico 6 (0.005555555924826792): peça, comprei, pra, necessidade, ser, vir, expectativa, atender, ter, produto
- Tópico 0 (0.2556332015292843): iphone, foto, 32, cartão, celular, película, memória, gb, livro, capa
- Tópico 5 (0.2597627919293714): chegar, comprei, enganoso, compr, foto, diferente, comprar, propaganda, vir, produto
- Tópico 18 (0.3901595603367901): solicitar, resposta, enviar, americanas, retorno, defeito, vir, troca, devolução, produto
